# PA Engine — weights + characteristics, 0CQ single frequency (PLACEHOLDER)

**Status: stub.** Call shape, auth, polling, grain splitting and landing are here and
correct against upstream PAEngine v3 (SDK 4.0.0). What is *not* here is anything
workstation-sourced — PA document path, component names, holdings account paths,
benchmark ids. Nothing runs until those are filled in via the discovery cell.

Companion to `spar_composite_returns_template.ipynb`. That one is **returns-based**
(SPAR, no holdings needed); this one is **holdings-based** (PA), which is what weights
and characteristics require.

## What this pulls

A point-in-time snapshot as of the most recent calendar quarter end — `frequency:
"Single"`, `enddate: "0CQ"`. Not a time series: one observation per holding or group at
one date.

| Tile | `componentdetail` | Returns |
|---|---|---|
| `weights` | **`GROUPSALL`** | group rows **and** security rows in one call |
| `characteristics` | `GROUPS` | group rows only |

### Why `GROUPSALL` for weights

`GROUPSALL` returns both levels in a single response, so one API call yields:

- **sector weights** — portfolio, benchmark, and active, from the group rows
- **security-level portfolio weights** — from the security rows, which is what a top-N
  holdings sort needs

That's the whole reason to prefer it over two calls at `GROUPS` and `SECURITIES`. The
component must actually expose portfolio weight, benchmark weight and active weight as
columns — `GROUPSALL` controls the row grain, not which columns exist. Check with Cell 4b
if active weight is missing.

**The cost is mixed grain in one payload.** Group rows and security rows arrive in the
same table with different meanings, and a sector row's weight is the sum of its
children's — so anything that sums the column without filtering by grain double-counts.
Cell 7 splits them into two tables (`pa_sector_weights`, `pa_security_weights`) rather
than landing one ambiguous one. The discriminator has to be confirmed against real output
on first run; see the note in that cell.

`componentdetail` accepts `GROUPS`, `GROUPSALL`, `TOTALS`, or `SECURITIES`.

## Units are grouped by benchmark, which is what makes multi-port unambiguous

`PACalculationParameters.benchmarks` is a **list**; SPAR's `benchmark` is a scalar. That
difference is what allows a multi-port call at all. But sending all four composites
against all three benchmarks in one unit would be ambiguous — with **4 accounts and 3
benchmarks the counts don't match, so positional pairing is impossible**, and the schema
doesn't say whether PA cross-products them or falls back to each account's default.

Grouping by benchmark sidesteps the question entirely — **one benchmark per unit**, so
there is nothing to pair:

| Unit | Accounts | Benchmark |
|---|---|---|
| `<tile>__r1000` | LC, LCS | Russell 1000 |
| `<tile>__r2500` | SMID | Russell 2500 |
| `<tile>__r3000` | CONC | Russell 3000 |

That's 3 units per tile, 6 total — still a genuine multi-port call (LC and LCS share one),
with no output filtering needed and no guesswork about which benchmark a row belongs to.

It also works here for a reason the SPAR notebook can't rely on: this is a **single-date
snapshot**, so every account in a unit shares one `dates` object. SPAR's
inception-to-date tiles each need a different `startdate` per strategy, and a unit carries
only one.

## Fee basis does not apply

Weights and characteristics are holdings attributes — there is no gross/net distinction.
The 2x fee-basis fan-out from the SPAR notebook is absent by design.

## Versions

| Package | Version |
|---|---|
| `fds.sdk.PAEngine` | **4.0.0** (upstream latest, 2026-07-21) |
| `fds.sdk.utils` | 3.0.1 |
| `fds.protobuf.stach.extensions` | 1.3.3 |
| `deltalake` | 1.6.2 |

**PAEngine 4.0.0 carries a breaking change beyond the 2026-05-20 Python-wide bump.** Per
upstream `BREAKING.md` (2026-07-21), `PADateParameters.enddate` and `.frequency` both lost
`required: true` and the schema's `required` array was dropped entirely, which
**reshuffles positional arguments** across ~6 operations including the calculation
endpoints and `convertPADatesToAbsoluteFormat`. A new optional `calendar` parameter was
also added.

The migration note is to use named parameters. Every call here passes keywords, so it is
unaffected — but older PA code passing dates positionally will silently bind the wrong
values, which is worth auditing separately.

> ⚠️ The PA SDK vendored under `code/python/PAEngine/v3/` in this repo is **2.2.2** and
> predates all of the above. Verify against upstream `main`.

Attach libraries to a **Fabric Environment**, not `%pip` — inline installs are disabled by
default in pipeline runs and unsupported in reference runs. Interactive first run only:
```
%pip install fds.sdk.PAEngine==4.0.0 fds.sdk.utils==3.0.1 \
             fds.protobuf.stach.extensions==1.3.3 deltalake==1.6.2
```

In [ ]:
# === Cell 1: credentials ===================================================
# Same HBCM_Config notebook as the SPAR pipeline — one place for the FactSet key.
%run HBCM_Config

In [ ]:
# === Cell 2: imports + API client ==========================================
import json, time, datetime as dt
import pandas as pd

import fds.sdk.PAEngine
from fds.sdk.PAEngine.api import (
    pa_calculations_api,
    components_api,
    accounts_api,
    columns_api,
    groups_api,
    frequencies_api,
    dates_api,          # PA HAS a DatesApi — SPAR does not. See Cell 4a.
)
from fds.sdk.PAEngine.models import (
    PACalculationParametersRoot,
    PACalculationParameters,
    PAIdentifier,
    PADateParameters,
    CalculationMeta,
)
from urllib3 import Retry

SDK_VERSION = fds.sdk.PAEngine.__version__
assert int(SDK_VERSION.split(".")[0]) >= 4, (
    f"fds.sdk.PAEngine {SDK_VERSION} found; this notebook targets >=4.0.0 "
    "(PADateParameters changed shape). Check the bound Fabric Environment."
)
print("PAEngine SDK", SDK_VERSION)

configuration = fds.sdk.PAEngine.Configuration(
    username=FACTSET_USER,
    password=FACTSET_APIKEY,
)
configuration.retries = Retry(
    total=3, status_forcelist=[500, 502, 503, 504], backoff_factor=2,
    allowed_methods=frozenset(["GET", "POST"]),
)

api_client = fds.sdk.PAEngine.ApiClient(configuration)
calc_api = pa_calculations_api.PACalculationsApi(api_client)
comp_api = components_api.ComponentsApi(api_client)

In [ ]:
# === Cell 3: THE CONFIG BLOCK ==============================================

CURRENCY = "USD"

# Point-in-time snapshot: one date, no series. Single frequency means startdate does not
# shape the result, but PA wants a coherent window, so it is set to the same date.
AS_OF_RELATIVE = "0CQ"
FREQUENCY = "Single"
USE_ABSOLUTE_AS_OF = False

def _prior_quarter_end(today=None):
    d = today or dt.date.today()
    qe = dt.date(d.year, ((d.month - 1) // 3) * 3 + 1, 1) - dt.timedelta(days=1)
    return qe.strftime("%Y%m%d")

AS_OF_ABS = _prior_quarter_end()
AS_OF = AS_OF_ABS if USE_ABSOLUTE_AS_OF else AS_OF_RELATIVE

# --- benchmarks ------------------------------------------------------------
# Same three groups as the SPAR notebook. Grouping accounts by benchmark is what keeps
# the multi-port call unambiguous — one benchmark per unit, nothing to pair.
BENCHMARK_GROUPS = {
    "r1000": {"label": "Russell 1000", "id": "<TODO>"},   # LC + LCS
    "r2500": {"label": "Russell 2500", "id": "<TODO>"},   # SMID
    "r3000": {"label": "Russell 3000", "id": "<TODO>"},   # CONC
}

# --- accounts --------------------------------------------------------------
# PA needs the HOLDINGS account path, not the returns ACCT the SPAR notebook uses — they
# are different objects, so do not copy one into the other.
# holdingsmode: B&H, TBR, OMS, EXT or VLT.
STRATEGIES = {
    "LC":   {"label": "Large Cap",           "acct": "<TODO path.ACCT>",
             "holdingsmode": "B&H", "bench_group": "r1000"},
    "SMID": {"label": "SMID",                "acct": "<TODO path.ACCT>",
             "holdingsmode": "B&H", "bench_group": "r2500"},
    "LCS":  {"label": "Large Cap Select",    "acct": "<TODO path.ACCT>",
             "holdingsmode": "B&H", "bench_group": "r1000"},
    "CONC": {"label": "Concentrated Equity", "acct": "<TODO path.ACCT>",
             "holdingsmode": "B&H", "bench_group": "r3000"},
}

# strategies per benchmark group, preserving declaration order
GROUP_MEMBERS = {}
for _c, _s in STRATEGIES.items():
    GROUP_MEMBERS.setdefault(_s["bench_group"], []).append(_c)

# --- tiles -----------------------------------------------------------------
# Resolved by name each run: a re-saved component can change id, and a stale id 400s with
# no useful message.
PA_DOCUMENT = "<TODO Client:/PA3/HBCM>"

TILES = {
    "weights": {
        "component_name": "<TODO exact workstation name>", "pinned_componentid": None,
        # GROUPSALL returns group rows AND security rows in one response, so sector
        # weights (port / bench / active) and security-level portfolio weights for top-N
        # sorting both come from this single call.
        "componentdetail": "GROUPSALL",
        "split_grain": True,
    },
    "characteristics": {
        "component_name": "<TODO exact workstation name>", "pinned_componentid": None,
        # GROUPS only. Switch to GROUPSALL if security-level characteristics are wanted
        # too — then set split_grain True here as well.
        "componentdetail": "GROUPS",
        "split_grain": False,
    },
}

# --- OneLake target --------------------------------------------------------
WORKSPACE_ID = "1b9fac18-9d75-4437-ab6c-b6ba44ff46a8"   # HBCM - Production
LAKEHOUSE_ID = "7cdf13b1-4586-4a02-b8ff-72fcf6db1277"   # hbcm_datahub
ONELAKE = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{LAKEHOUSE_ID}"
RAW_DIR = f"{ONELAKE}/Files/raw/pa"
# Separate tables per grain — a sector row's weight is the sum of its securities', so one
# shared table would double-count on any unfiltered SUM.
TABLE_SECTOR = f"{ONELAKE}/Tables/factset/pa_sector_weights"
TABLE_SECURITY = f"{ONELAKE}/Tables/factset/pa_security_weights"
TABLE_CHARACTERISTICS = f"{ONELAKE}/Tables/factset/pa_characteristics"

assert all(s["bench_group"] in BENCHMARK_GROUPS for s in STRATEGIES.values())
print(f"as-of sent: {AS_OF!r}  frequency: {FREQUENCY}  asof_date written: {AS_OF_ABS}")
print(f"{len(TILES)} tiles x {len(GROUP_MEMBERS)} benchmark groups = "
      f"{len(TILES) * len(GROUP_MEMBERS)} units")
for g, members in GROUP_MEMBERS.items():
    print(f"  {g:<8} {BENCHMARK_GROUPS[g]['label']:<14} <- {', '.join(members)}")

In [ ]:
# === Cell 4: resolve dates + component ids, every run =====================

# (a) Resolve 0CQ server-side and compare against the locally computed quarter end.
#     PA can do this; SPAR has no DatesApi and cannot. If they disagree, the asof_date
#     label would be wrong — this is the check the SPAR notebook structurally can't run.
d_api = dates_api.DatesApi(api_client)
try:
    resolved_dates = d_api.convert_pa_dates_to_absolute_format(
        startdate=AS_OF, enddate=AS_OF,
    )
    print("PA resolved dates:", resolved_dates)
    print(f"locally computed quarter end: {AS_OF_ABS}")
except Exception as e:
    # 4.0.0 reshuffled this signature and added an optional `calendar` parameter. If this
    # errors, check the current arg list before concluding 0CQ is wrong.
    print(f"date conversion unavailable ({e!r}); relying on AS_OF_ABS for labelling")

# (b) component name -> live id
def _field(obj, name):
    if hasattr(obj, name):
        return getattr(obj, name)
    try:
        return obj.get(name)
    except AttributeError:
        return None

def resolve_component_ids(document=PA_DOCUMENT):
    summary = comp_api.get_pa_components(document=document)
    by_name = {}
    for cid, meta in (summary.data or {}).items():
        by_name.setdefault(_field(meta, "name"), []).append(cid)

    resolved, drift, missing = {}, [], []
    for tile, cfg in TILES.items():
        hits = by_name.get(cfg["component_name"], [])
        if len(hits) != 1:
            missing.append((tile, cfg["component_name"], len(hits)))
            continue
        resolved[tile] = hits[0]
        if cfg.get("pinned_componentid") and cfg["pinned_componentid"] != hits[0]:
            drift.append((tile, cfg["component_name"], cfg["pinned_componentid"], hits[0]))
    return resolved, drift, missing, by_name

RESOLVED_COMPONENTS, COMPONENT_DRIFT, COMPONENT_MISSING, COMPONENTS_BY_NAME = \
    resolve_component_ids()

for tile, cid in RESOLVED_COMPONENTS.items():
    print(f"{tile:<18} {cid}  ({TILES[tile]['component_name']})")

if COMPONENT_DRIFT:
    print("\n*** COMPONENT ID DRIFT — the component was re-saved. Confirm its columns")
    print("*** still match, then update pinned_componentid:")
    for tile, name, was, now in COMPONENT_DRIFT:
        print(f"    {tile}: {name!r}  {was} -> {now}")

if COMPONENT_MISSING:
    print("\nUnresolved tiles:", COMPONENT_MISSING)
    print("Available component names:")
    for name, cids in sorted(COMPONENTS_BY_NAME.items(), key=lambda kv: str(kv[0])):
        print(f"    {name!r}: {cids}")

assert not COMPONENT_MISSING, "every tile needs exactly one matching component name"

In [ ]:
# === Cell 4b: DISCOVERY — run once interactively ==========================
# Fills the TODOs in Cell 3. Not part of the scheduled path.

a_api = accounts_api.AccountsApi(api_client)
col_api = columns_api.ColumnsApi(api_client)
grp_api = groups_api.GroupsApi(api_client)
frq_api = frequencies_api.FrequenciesApi(api_client)

# Holdings account paths -> STRATEGIES[...]["acct"]
#   print(a_api.get_accounts(path="Client:/"))

# Weight columns. GROUPSALL sets the row grain, NOT which columns exist — if active
# weight is missing from the output, the component doesn't expose it and either the
# component needs editing or `columns` needs overriding on the request.
#   print(col_api.get_pa_columns(name="weight", category="", directory=""))

# Groupings -> what the sector rows will actually be grouped by. PA uses the component's
# saved grouping unless PACalculationParameters.groups overrides it.
#   print(grp_api.get_pa_groups())

# Confirm "Single" is spelled as PA expects
#   print(frq_api.get_pa_frequencies())

# Confirm each Russell benchmark id resolves
#   print(comp_api.get_pa_component_by_id(id=RESOLVED_COMPONENTS["weights"]))
print("uncomment the lookup you need")

In [ ]:
# === Cell 5: build units — one per (tile, benchmark group) ================
# Accounts sharing a benchmark travel together, so each unit has exactly ONE benchmark.
# That removes the account<->benchmark pairing question entirely: with 4 accounts and 3
# benchmarks in a single unit, positional pairing is arithmetically impossible and the
# schema doesn't say whether PA cross-products or falls back to account defaults. One
# benchmark per unit means there is nothing to infer and no output to filter.
#
# Safe here only because this is a single-date snapshot — every account in a unit shares
# one `dates` object. That is exactly the constraint that blocks the same collapse in the
# SPAR notebook's inception-to-date tiles.

def pa_dates() -> PADateParameters:
    # Keyword args throughout: 4.0.0 dropped `required` on enddate/frequency, which
    # reshuffled positional arguments across the PA calculation endpoints.
    return PADateParameters(startdate=AS_OF, enddate=AS_OF, frequency=FREQUENCY)

def build_unit(tile_name: str, tile_cfg: dict, group: str) -> PACalculationParameters:
    return PACalculationParameters(
        componentid=RESOLVED_COMPONENTS[tile_name],
        accounts=[
            PAIdentifier(id=STRATEGIES[c]["acct"],
                         holdingsmode=STRATEGIES[c]["holdingsmode"])
            for c in GROUP_MEMBERS[group]
        ],
        benchmarks=[PAIdentifier(id=BENCHMARK_GROUPS[group]["id"])],
        dates=pa_dates(),
        currencyisocode=CURRENCY,
        componentdetail=tile_cfg["componentdetail"],
    )

UNIT_KEYS, units = {}, {}
for tile_name, tile_cfg in TILES.items():
    for group in GROUP_MEMBERS:
        key = f"{tile_name}__{group}"
        units[key] = build_unit(tile_name, tile_cfg, group)
        UNIT_KEYS[key] = (tile_name, group)

params_root = PACalculationParametersRoot(
    data=units,
    meta=CalculationMeta(
        contentorganization="SimplifiedRow",
        stach_content_organization="SimplifiedRow",
        contenttype="Json",
        format="JsonStach",
    ),
)
for key, (tile_name, group) in UNIT_KEYS.items():
    print(f"{key:<28} {TILES[tile_name]['componentdetail']:<10} "
          f"{BENCHMARK_GROUPS[group]['label']:<14} <- {', '.join(GROUP_MEMBERS[group])}")

In [ ]:
# === Cell 6: submit + poll ================================================
# Identical semantics to SPAR: 200 sync, 201 ready, 202 poll. Multi-unit always 202.

def run_pa(params_root, deadline=10, poll_interval=3, timeout=900):
    wrapper = calc_api.post_and_calculate(
        x_fact_set_api_long_running_deadline=deadline,
        pa_calculation_parameters_root=params_root,
    )
    code = wrapper.get_status_code()
    if code == 200:
        status_root = wrapper.get_response_200()
    elif code == 201:
        status_root = wrapper.get_response_201()
    elif code == 202:
        status_root = wrapper.get_response_202()
        calc_id = status_root.data.calculationid
        deadline_at = time.time() + timeout
        while True:
            if time.time() > deadline_at:
                calc_api.cancel_calculation_by_id(id=calc_id)
                raise TimeoutError(f"calc {calc_id} exceeded {timeout}s (cancelled)")
            poll = calc_api.get_calculation_status_by_id(id=calc_id)
            if poll.get_status_code() == 200:
                status_root = poll.get_response_200()
                break
            if poll.get_status_code() != 202:
                raise RuntimeError(f"unexpected poll status {poll.get_status_code()}")
            time.sleep(poll_interval)
    else:
        raise RuntimeError(f"unexpected submit status {code}")

    calc_id = status_root.data.calculationid
    out = []
    for unit_id, unit_status in (status_root.data.units or {}).items():
        st = getattr(unit_status, "status", None)
        if st != "Success":
            out.append((unit_id, None, st))
            continue
        out.append((unit_id,
                    calc_api.get_calculation_unit_result_by_id(id=calc_id, unit_id=unit_id),
                    st))
    return calc_id, out

calc_id, results = run_pa(params_root)
failed = [(u, s) for u, r, s in results if r is None]
print(f"calc={calc_id} ok={len(results) - len(failed)} failed={len(failed)}")
for u, s in failed:
    print(f"  FAILED {u}: {s}")
assert results and not failed, "resolve failures before writing to the lakehouse"

In [ ]:
# === Cell 7: raw landing + STACH -> DataFrame ==============================
from fds.protobuf.stach.extensions.StachExtensionFactory import StachExtensionFactory
from fds.protobuf.stach.extensions.StachVersion import StachVersion

asof_tag = AS_OF_ABS
for unit_id, res, _ in results:
    notebookutils.fs.put(f"{RAW_DIR}/asof={asof_tag}/{unit_id}.json",
                         json.dumps(res.to_dict(), default=str), True)
print(f"landed {len(results)} raw payloads")

def stach_to_dataframes(api_response):
    ext = StachExtensionFactory.get_stach_extension(StachVersion.V2)
    return [pd.DataFrame(t.data, columns=t.columns)
            for t in ext.convert(json.dumps(api_response.to_dict(), default=str))]

frames = []
for unit_id, res, _ in results:
    tile_name, group = UNIT_KEYS[unit_id]
    for i, df in enumerate(stach_to_dataframes(res)):
        df = df.copy()
        for pos, (col, val) in enumerate([
            ("asof_date",       asof_tag),
            ("tile",            tile_name),
            ("bench_group",     group),
            ("benchmark_id",    BENCHMARK_GROUPS[group]["id"]),
            ("benchmark_label", BENCHMARK_GROUPS[group]["label"]),
            ("componentid",     RESOLVED_COMPONENTS[tile_name]),
            ("componentdetail", TILES[tile_name]["componentdetail"]),
            ("table_ix",        i),
        ]):
            df.insert(pos, col, val)
        frames.append(df)

raw_tidy = pd.concat(frames, ignore_index=True)
raw_tidy.columns = [str(c).strip().replace(" ", "_").lower() for c in raw_tidy.columns]

print(raw_tidy.shape)
print("\ncolumns:", list(raw_tidy.columns))
print("\nrows per unit:")
print(raw_tidy.groupby(["tile", "bench_group", "table_ix"]).size())

# --- TWO THINGS TO RESOLVE FROM THIS OUTPUT BEFORE THE WRITE ---------------
#
# 1. WHICH ACCOUNT IS EACH ROW?
#    A unit can hold two accounts (LC + LCS), so the strategy is NOT recoverable from the
#    unit key. It has to come from a portfolio/account column in the output. Find it, then
#    fill ACCT_TO_CODE and set STRATEGY_COL.
#
# 2. WHICH ROWS ARE GROUPS AND WHICH ARE SECURITIES?
#    GROUPSALL interleaves both grains. STACH usually distinguishes them by a hierarchy
#    level/depth column, or by the security identifier being null on group rows. The
#    discriminator is component-dependent and cannot be known in advance — inspect the
#    columns printed above and implement is_group_row() accordingly.
#
#    Getting this wrong is not a cosmetic problem: a sector row's weight is the sum of its
#    children's, so mixing grains double-counts every total.

STRATEGY_COL = None          # e.g. "portfolio" / "account" — set from the columns above
ACCT_TO_CODE = {s["acct"]: c for c, s in STRATEGIES.items()}

def attach_strategy(df):
    if STRATEGY_COL is None or STRATEGY_COL not in df.columns:
        df["strategy_code"] = pd.NA
        df["strategy"] = pd.NA
        return df
    df["strategy_code"] = df[STRATEGY_COL].map(ACCT_TO_CODE)
    df["strategy"] = df["strategy_code"].map(
        {c: s["label"] for c, s in STRATEGIES.items()})
    return df

def is_group_row(df):
    """True for sector/group rows, False for security rows. HEURISTIC — confirm it."""
    for col in ("level", "depth", "hierarchy_level"):
        if col in df.columns:
            return pd.to_numeric(df[col], errors="coerce") == 0
    for col in ("security", "security_id", "asset_id", "symbol", "ticker"):
        if col in df.columns:
            return df[col].isna() | (df[col].astype("string").str.strip() == "")
    return None              # no discriminator found — do not guess

tidy = attach_strategy(raw_tidy)
print("\nSTRATEGY_COL:", STRATEGY_COL,
      "| strategy_code populated:", int(tidy["strategy_code"].notna().sum()),
      "of", len(tidy))
_mask = is_group_row(tidy)
print("grain discriminator found:", _mask is not None,
      f"| group rows: {int(_mask.sum())}" if _mask is not None else "")
display(tidy.head(30))

In [ ]:
# === Cell 8: split by grain and write ======================================
# Disabled until Cell 7's two unknowns are resolved. Writing a table whose strategy is
# null on every row, or whose grains are mixed, is worse than not writing — both look
# loaded downstream, and the mixed-grain one silently double-counts.

# from deltalake import DeltaTable, write_deltalake
#
# assert STRATEGY_COL is not None, "set STRATEGY_COL in Cell 7"
# assert tidy["strategy_code"].notna().all(), \
#     "some rows did not map to a strategy — check ACCT_TO_CODE against the output values"
#
# def write(path, df, name):
#     if df.empty:
#         print(f"skip {name}: no rows")
#         return
#     df = df.astype({c: "string" for c in df.select_dtypes("object").columns})
#     try:
#         DeltaTable(path).delete(f"asof_date = '{asof_tag}'")
#         mode = "append"
#     except Exception:
#         mode = "overwrite"
#     write_deltalake(path, df, mode=mode, schema_mode="merge")
#     print(f"wrote {len(df)} rows to {name} (mode={mode})")
#
# weights = tidy[tidy["tile"] == "weights"]
# chars = tidy[tidy["tile"] == "characteristics"]
#
# mask = is_group_row(weights)
# assert mask is not None, "implement is_group_row() for this component's output"
# write(TABLE_SECTOR, weights[mask], "factset.pa_sector_weights")
# write(TABLE_SECURITY, weights[~mask], "factset.pa_security_weights")
# write(TABLE_CHARACTERISTICS, chars, "factset.pa_characteristics")

print("write step disabled — this notebook is a placeholder")

## To finish this notebook

1. Fill the Cell 3 TODOs: `PA_DOCUMENT`, the two `component_name`s, each strategy's
   holdings `acct` path, and the three Russell benchmark ids. Cell 4b's lookups find them.
2. Confirm `holdingsmode`. `B&H` is the default guess; `TBR`, `OMS`, `EXT` and `VLT` are
   the alternatives, and the right one depends on how the composites are maintained.
3. **Set `STRATEGY_COL`** (Cell 7) to whichever output column carries the account, and
   check `ACCT_TO_CODE` maps its actual values. Units hold up to two accounts, so the
   strategy cannot come from the unit key.
4. **Implement `is_group_row()`** (Cell 7) against the real `GROUPSALL` output. The
   supplied version guesses at a level/depth column, then at a null security identifier;
   confirm which actually applies. This is the one that quietly corrupts numbers if wrong.
5. Confirm the weights component exposes **active** weight, not just portfolio and
   benchmark. `GROUPSALL` sets the row grain, not the column set — if active weight is
   absent, edit the component or override `columns` on the request.
6. Uncomment the write in Cell 8.

## Open questions

- **Does `Single` frequency at `0CQ` return holdings *as of* the quarter end, or the
  quarter's activity?** Determines whether weights are a snapshot or a period average.
  Worth checking against the workstation for one composite before trusting it.
- **Are the sector rows grouped the way you want?** PA uses the component's saved
  grouping unless `PACalculationParameters.groups` overrides it, so the sector scheme is
  whatever was configured in the workstation.
- **Do SPAR and PA need to reconcile?** They run against different account objects — a
  returns ACCT vs a holdings account — so agreement is not automatic and a difference is
  not necessarily an error.

## Sources

Verified against **upstream `FactSet/enterprise-sdk` `main`** (PAEngine v3, SDK 4.0.0) —
`PACalculationParameters.md` (`accounts`/`benchmarks` as lists, `componentdetail` values
`GROUPS`/`GROUPSALL`/`TOTALS`/`SECURITIES`, `columns`, `groups`), `PADateParameters.md`,
`PAIdentifier.md` (`holdingsmode` values), `PACalculationsApi.md`, `ComponentsApi.md`,
`DatesApi.md`, `ColumnsApi.md`, `GroupsApi.md`, `FrequenciesApi.md`, and `BREAKING.md`
(2026-07-21 `PADateParameters`; 2026-05-20 Python-wide bump).

Not against `code/python/PAEngine/v3/` in this repo, which is pinned at 2.2.2 and
predates the 4.0.0 changes.